# ScaleFlow: Real Multiscale OT-CFM Training on Dual NVIDIA T4 (Kaggle)
### Full Text-to-Image Training at 512x512 Resolution (3 Epochs)

This notebook implements complete, real-world training of **ScaleFlow** using:
- **Hardware:** 2x NVIDIA Tesla T4 (16GB VRAM each = 32GB total) with `torch.nn.DataParallel` and **Automatic Mixed Precision (AMP float16)**.
- **Datasets:**
  1. `prithivMLmods/Caption3o-Opt-v2` (~10k descriptive visual captions)
  2. `maderix/flickr_bw_rgb` (~7.2k photography captions)
  - Combined total: ~17,000 high-quality image-caption pairs.
- **Resolution:** Native $512 \times 512$ (`max_scale=6`), stepping through multiscale pyramid transitions:
  $$8 \to 16 \to 32 \to 64 \to 128 \to 256 \to 512$$
- **Model Size:** Production-grade configuration (~42M parameters) with Hybrid Conv/Attention Text Encoder and AdaLN ConvNeXt V2 Decoder.
- **Training Duration:** 3 full epochs with cosine learning rate schedule and checkpointing.


In [ ]:
# ==========================================
# 1. VERIFY HARDWARE ACCELERATION (DUAL T4)
# ==========================================
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

gpu_count = torch.cuda.device_count()
print(f"GPU Count:       {gpu_count}")

if gpu_count < 2:
    print("WARNING: Expected 2 GPUs in Kaggle Settings (Accelerator: GPU T4 x2).")
    print("Make sure you select 'GPU T4 x2' in the Kaggle sidebar settings!")

for i in range(gpu_count):
    prop = torch.cuda.get_device_properties(i)
    vram_gb = prop.total_memory / (1024 ** 3)
    print(f"  [GPU {i}]: {prop.name} | Total VRAM: {vram_gb:.2f} GB")


In [ ]:
# ==========================================
# 2. INSTALL SCALEFLOW & DEPENDENCIES
# ==========================================
!pip install -q git+https://github.com/bueormnew/scaleflow.git
!pip install -q datasets pillow torchvision tqdm matplotlib accelerate

import scaleflow
print(f"Successfully loaded ScaleFlow version {scaleflow.__version__}")


In [ ]:
# ==========================================
# 3. IMPORTS AND GLOBAL CONFIGURATION
# ==========================================
import random
import math
import time
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from datasets import load_dataset, concatenate_datasets

from scaleflow.models import ScaleFlowModel, ScaleFlowConfig
from scaleflow.tokenizer import BPETokenizer
from scaleflow.training import (
    MultiscaleOTFlowMatching,
    FlowMatchingLoss,
    VelocityErrorMetric,
    build_image_pyramid,
    DEFAULT_SCALE_RESOLUTIONS,
)
from scaleflow.pipeline import ScaleFlowPipeline, tensor_to_pil

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path("/kaggle/working/scaleflow_output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")


In [ ]:
# ==========================================
# 4. LOAD & COMBINE DATASETS (~17k SAMPLES)
# ==========================================
print("Loading 'prithivMLmods/Caption3o-Opt-v2'...")
ds_caption3o = load_dataset("prithivMLmods/Caption3o-Opt-v2", split="train")
print(f"  -> Caption3o samples: {len(ds_caption3o):,}")

print("Loading 'maderix/flickr_bw_rgb'...")
ds_flickr = load_dataset("maderix/flickr_bw_rgb", split="train")
print(f"  -> Flickr samples:    {len(ds_flickr):,}")

# Unify columns: ensure both have 'image' and 'caption'
def standardize_features(ds):
    cols = ds.column_names
    img_col = "image" if "image" in cols else "img"
    cap_col = "caption" if "caption" in cols else "text"
    
    # Keep only the two required columns
    return ds.select_columns([img_col, cap_col]).rename_columns({img_col: "image", cap_col: "caption"})

ds_caption3o = standardize_features(ds_caption3o)
ds_flickr = standardize_features(ds_flickr)

# Combine datasets
combined_dataset = concatenate_datasets([ds_caption3o, ds_flickr])
print(f"\nTotal combined training dataset: {len(combined_dataset):,} samples")

# Preview sample
sample_0 = combined_dataset[0]
print(f"Sample 0 caption: {sample_0['caption']}")
print(f"Sample 0 image format: {sample_0['image'].format}, size: {sample_0['image'].size}")


In [ ]:
# ==========================================
# 5. BUILD & FIT DOMAIN BPE TOKENIZER
# ==========================================
TOKENIZER_PATH = OUTPUT_DIR / "tokenizer.json"
VOCAB_SIZE = 4096

print(f"Training BPETokenizer on dataset captions (Vocab size: {VOCAB_SIZE})...")

# Extract text corpus from a representative slice of captions
corpus = [sample["caption"] for sample in combined_dataset if isinstance(sample["caption"], str) and len(sample["caption"].strip()) > 0]

tokenizer = BPETokenizer()
tokenizer.train_from_corpus(corpus, vocab_size=VOCAB_SIZE, min_frequency=2)
tokenizer.save(TOKENIZER_PATH)

print(f"Tokenizer trained with {len(tokenizer.vocab):,} vocabulary items and saved to {TOKENIZER_PATH}")

# Quick sanity check
test_text = "A cinematic shot of a vibrant sunset over mountain peaks"
encoded = tokenizer(test_text, padding=True, return_tensors="pt")
decoded = tokenizer.decode(encoded["input_ids"][0].tolist())
print(f"Test encoded shape: {encoded['input_ids'].shape}")
print(f"Decoded text:       '{decoded}'")


In [ ]:
# ==========================================
# 6. PYTORCH DATASET & DATALOADER (512x512)
# ==========================================
class HuggingFaceMultiscaleDataset(Dataset):
    """Adapts HuggingFace image-caption datasets to ScaleFlow 512x512 format."""
    
    def __init__(self, hf_dataset, tokenizer, image_size=512, max_seq_len=64):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.image_size = image_size
        self.max_seq_len = max_seq_len
        
        # Clean 512x512 standard image transformation
        self.transform = T.Compose([
            T.Resize(image_size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # Normalized to [-1, 1]
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        raw_img = item["image"]
        caption = str(item.get("caption", ""))
        
        # Ensure RGB 3-channel (handles grayscale from flickr_bw_rgb or RGBA)
        if raw_img.mode != "RGB":
            raw_img = raw_img.convert("RGB")
            
        img_tensor = self.transform(raw_img)
        
        # Tokenize caption with fixed padding
        encoded = self.tokenizer.encode(caption)
        pad_id = getattr(self.tokenizer, "pad_token_id", 0)
        
        if len(encoded) > self.max_seq_len:
            encoded = encoded[:self.max_seq_len]
            mask = [1] * self.max_seq_len
        else:
            mask = [1] * len(encoded) + [0] * (self.max_seq_len - len(encoded))
            encoded = encoded + [pad_id] * (self.max_seq_len - len(encoded))
            
        return {
            "image": img_tensor,
            "input_ids": torch.tensor(encoded, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
            "caption": caption,
        }

def collate_fn(batch):
    images = torch.stack([x["image"] for x in batch], dim=0)
    input_ids = torch.stack([x["input_ids"] for x in batch], dim=0)
    attention_mask = torch.stack([x["attention_mask"] for x in batch], dim=0)
    captions = [x["caption"] for x in batch]
    return {
        "images": images,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "captions": captions,
    }

train_dataset = HuggingFaceMultiscaleDataset(
    combined_dataset,
    tokenizer=tokenizer,
    image_size=512,
    max_seq_len=64
)

# Batch size: 8 per GPU x 2 GPUs = 16 effective batch size for 512x512
BATCH_SIZE = 16
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    collate_fn=collate_fn,
)

print(f"DataLoader created: {len(train_loader):,} batches per epoch (Batch size: {BATCH_SIZE})")


In [ ]:
# ==========================================
# 7. INITIALIZE PRODUCTION SCALEFLOW MODEL
# ==========================================
# Scale 6 corresponds to 512x512 resolution (scale 0=8, 1=16, 2=32, 3=64, 4=128, 5=256, 6=512)
model_config = ScaleFlowConfig(
    vocab_size=len(tokenizer.vocab),
    text_dim=512,                  # High capacity text representation
    text_depth=6,                  # 6 Hybrid Conv/Attention blocks
    text_heads=8,
    max_seq_len=64,
    
    max_scale=6,                   # Native 512x512 support
    use_scale_mlp=True,
    
    in_channels=3,
    out_channels=3,
    decoder_dim=384,               # 384-channel ConvNeXt V2 Backbone
    decoder_depth=6,               # 6 deep residual stages with AdaLN & GRN
    decoder_mlp_ratio=4.0,
    
    use_cross_attention=True,
    cross_attention_max_scale=3,   # Spatial cross-attention at <= 64x64, bypass at high-res
    cross_attention_heads=8,
)

model = ScaleFlowModel(model_config)

param_count = model.num_parameters()
print(f"Model successfully built!")
print(f"Total Parameters:    {param_count['total']:,} ({param_count['total'] / 1e6:.2f} M)")
print(f"Trainable Parameters:{param_count['trainable']:,}")
print(f"Text Encoder:        {param_count['text_encoder']:,} params")
print(f"Decoder Backbone:    {param_count['decoder']:,} params")


In [ ]:
# ==========================================
# 8. SETUP MULTISCALE OPTIMAL TRANSPORT CFM
# ==========================================
# Scale mapping: 0: 8x8, 1: 16x16, 2: 32x32, 3: 64x64, 4: 128x128, 5: 256x256, 6: 512x512
SCALE_RESOLUTIONS = {
    0: 8,
    1: 16,
    2: 32,
    3: 64,
    4: 128,
    5: 256,
    6: 512,
}

flow_matcher = MultiscaleOTFlowMatching(
    scale_resolutions=SCALE_RESOLUTIONS,
    sigma_min=1e-4,
    noise_prev_scale_sigma=0.5,
    prev_scale_noise_mode="blend",
    loss_type="mse"
)

# Loss function with balanced multi-scale weighting
loss_fn = FlowMatchingLoss(loss_type="mse")
metric = VelocityErrorMetric()

print("Multiscale OT-CFM trajectory generator ready (Resolutions: 8x8 -> 512x512).")


In [ ]:
# ==========================================
# 9. DUAL T4 HIGH-THROUGHPUT TRAINING LOOP
# ==========================================
EPOCHS = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Multi-GPU Parallelization via DataParallel across both T4 GPUs
if torch.cuda.device_count() > 1:
    print(f"Enabling torch.nn.DataParallel across {torch.cuda.device_count()} GPUs: {[0, 1]}")
    parallel_model = nn.DataParallel(model, device_ids=[0, 1])
else:
    parallel_model = model

# AdamW with weight decay separation
decay_params, no_decay_params = [], []
no_decay_names = ("bias", "norm", "gamma", "beta", "ada_proj")

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(nd in name.lower() for nd in no_decay_names) or param.ndim <= 1:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.99), eps=1e-8)

# Cosine Learning Rate Schedule with Warmup
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Automatic Mixed Precision (AMP) Scaler
scaler = torch.cuda.amp.GradScaler()

print(f"Training Plan: {EPOCHS} Epochs | {total_steps:,} Steps | Warmup: {warmup_steps} Steps | AMP: True")


In [ ]:
# Execute 3 Epochs of Multiscale OT-CFM Training
history = []
global_step = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    parallel_model.train()
    epoch_loss = 0.0
    metric.reset()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)
    
    for batch_idx, batch in enumerate(pbar):
        images = batch["images"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        B = images.shape[0]
        
        optimizer.zero_grad(set_to_none=True)
        
        # Build multi-resolution pyramid on GPU (8 -> 512)
        pyramid = flow_matcher.build_pyramid(images)
        
        # Sample random scale level s in [0..6] (8x8 to 512x512)
        scale = flow_matcher.sample_scale(batch_size=B, min_scale=0, max_scale=6, per_batch=True)
        flow_batch = flow_matcher.prepare_flow_batch(pyramid=pyramid, scale_level=scale)
        
        # Direct velocity prediction on current noisy state x_t
        with torch.cuda.amp.autocast():
            v_pred = parallel_model(
                flow_batch.x_t,
                scale=scale,
                input_ids=input_ids,
                attention_mask=attention_mask,
                upsample=False
            )
            loss = loss_fn(v_pred, flow_batch.velocity_target)
            
        # AMP Scaled Backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # Tracking metrics
        loss_val = loss.item()
        epoch_loss += loss_val
        global_step += 1
        
        metric_res = metric.update(v_pred=v_pred.detach(), v_target=flow_batch.velocity_target, scale_level=scale)
        current_lr = optimizer.param_groups[0]["lr"]
        
        pbar.set_postfix({
            "loss": f"{loss_val:.4f}",
            "mse": f"{metric_res['mse']:.4f}",
            "cos_sim": f"{metric_res['cosine_sim']:.3f}",
            "scale": scale,
            "lr": f"{current_lr:.2e}"
        })
        
        if global_step % 20 == 0:
            history.append({
                "step": global_step,
                "epoch": epoch,
                "loss": loss_val,
                "lr": current_lr,
                "mse": metric_res["mse"],
                "cos_sim": metric_res["cosine_sim"]
            })
            
    # Save checkpoint at the end of each epoch
    epoch_ckpt_path = CHECKPOINT_DIR / f"checkpoint_epoch_{epoch}.pt"
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": model_config.to_dict(),
    }, epoch_ckpt_path)
    print(f"--> Saved checkpoint for Epoch {epoch} to: {epoch_ckpt_path}")

total_elapsed = time.time() - start_time
print(f"\nAll {EPOCHS} epochs completed in {total_elapsed/60:.2f} minutes!")

# Save final pretrained model package (loadable via ScaleFlowPipeline.from_pretrained)
FINAL_MODEL_DIR = OUTPUT_DIR / "scaleflow_512_final"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save(FINAL_MODEL_DIR / "tokenizer.json")
print(f"Final deployable model package saved to: {FINAL_MODEL_DIR}")


In [ ]:
# ==========================================
# 10. PLOT TRAINING CURVES
# ==========================================
if history:
    steps = [h["step"] for h in history]
    losses = [h["loss"] for h in history]
    cos_sims = [h["cos_sim"] for h in history]
    lrs = [h["lr"] for h in history]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(steps, losses, color="#2563eb", lw=1.5)
    axes[0].set_title("Flow Matching MSE Loss")
    axes[0].set_xlabel("Global Steps")
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(steps, cos_sims, color="#16a34a", lw=1.5)
    axes[1].set_title("Velocity Cosine Similarity")
    axes[1].set_xlabel("Global Steps")
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(steps, lrs, color="#dc2626", lw=1.5)
    axes[2].set_title("Learning Rate Schedule")
    axes[2].set_xlabel("Global Steps")
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=200)
    plt.show()


In [ ]:
# ==========================================
# 11. INFERENCE & 512x512 SAMPLE GENERATION
# ==========================================
from IPython.display import display

print(f"Loading trained pipeline from: {FINAL_MODEL_DIR}...")
pipeline = ScaleFlowPipeline.from_pretrained(FINAL_MODEL_DIR)
pipeline = pipeline.to("cuda")

eval_prompts = [
    "A majestic snow-capped mountain reflecting in a pristine alpine lake, golden hour lighting",
    "A cozy warm coffee shop interior with raindrops on the window, soft cinematic bokeh",
    "A futuristic cyberpunk street with glowing neon signs and reflective wet pavement",
    "Colorized photograph of a vintage steam locomotive crossing an old stone bridge"
]

print("Generating 512x512 images using 6 recursive scale steps (8 -> 16 -> 32 -> 64 -> 128 -> 256 -> 512)...")

generated_images = []
for i, prompt in enumerate(eval_prompts):
    print(f"\nGenerating [{i+1}/{len(eval_prompts)}]: '{prompt}'")
    
    # steps=6 generates from 8x8 up to 512x512
    img = pipeline(
        prompt=prompt,
        steps=6,
        seed=100 + i,
        guidance_scale=2.5,
        output_type="pil"
    )
    
    save_path = OUTPUT_DIR / f"generated_sample_{i+1}.png"
    img.save(save_path)
    generated_images.append(img)
    print(f"Saved: {save_path} (Size: {img.size})")
    display(img)


In [ ]:
# ==========================================
# 12. ARCHIVE CHECKPOINTS FOR EASY DOWNLOAD
# ==========================================
!cd /kaggle/working && zip -r -q scaleflow_512_trained.zip scaleflow_output/scaleflow_512_final scaleflow_output/*.png
print("Archive created: /kaggle/working/scaleflow_512_trained.zip")
print("You can download it directly from the Kaggle Output section on the right panel!")
